In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 7 Lab — End-to-End Pipelines and Visualization
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Wrapping preprocessing and model into a single `Pipeline`
- Tuning the full pipeline with `GridSearchCV` using `step__param` naming
- Extracting predicted probabilities and segmenting customers into risk tiers
- Communicating results with interactive Plotly Express charts

**Estimated time:** 85–100 minutes

---

## The Dataset

This lab uses the same customer churn dataset from Weeks 5 and 6. By Week 7, you know this data well — the goal now is to wrap the full workflow into a clean, deployment-ready pipeline and present the results as business insight.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import classification_report, f1_score, recall_score

# ── Rebuild churn dataset ───────────────────────────────────────────────────
np.random.seed(13)
n = 400

tenure       = np.random.randint(1, 73, n)
monthly_chg  = np.round(np.random.uniform(20, 120, n), 2)
num_products = np.random.randint(1, 6, n)
tech_support = np.random.choice([0, 1], n, p=[0.45, 0.55])
contract     = np.random.choice(
    ['Month-to-Month', 'One Year', 'Two Year'], n, p=[0.50, 0.30, 0.20]
)
segment      = np.random.choice(
    ['Residential', 'SMB', 'Enterprise'], n, p=[0.55, 0.30, 0.15]
)
contract_risk = np.where(contract == 'Month-to-Month', 0.35,
               np.where(contract == 'One Year', 0.10, 0.03))
segment_risk  = np.where(segment == 'Residential', 0.08,
               np.where(segment == 'SMB', 0.03, 0.00))
churn_prob = np.clip(
    0.45 - (tenure / 72) * 0.35
    + (monthly_chg - 20) / 100 * 0.20
    - tech_support * 0.10
    - (num_products - 1) / 4 * 0.08
    + contract_risk + segment_risk,
    0.02, 0.95
)
churned = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

df = pd.DataFrame({
    'tenure_months':    tenure,
    'monthly_charges':  monthly_chg,
    'num_products':     num_products,
    'tech_support':     tech_support,
    'contract_type':    contract,
    'customer_segment': segment,
    'churned':          churned,
})

X = df.drop(columns=['churned'])
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Dataset ready: {df.shape[0]} rows  |  Churn rate: {y.mean():.1%}')
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

---
## Part 1 — Building the Pipeline

A `Pipeline` chains a `ColumnTransformer` and a model into one object. `.fit()` runs every step in order. `.predict()` transforms then classifies — automatically, in the right sequence, with no risk of fitting the preprocessor on test data.

In [ ]:
# Define the preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('ord',  OrdinalEncoder(categories=[['Month-to-Month', 'One Year', 'Two Year']]),
             ['contract_type']),
    ('cat',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
             ['customer_segment']),
    ('num',  StandardScaler(),
             ['tenure_months', 'monthly_charges', 'num_products']),
    ('pass', 'passthrough',
             ['tech_support']),
])

# Build the pipeline
pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model',        RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Fit on raw X_train — no separate transform needed
pipe.fit(X_train, y_train)

# Predict on raw X_test
y_pred = pipe.predict(X_test)

print('Pipeline — Test Set:')
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

---
### 🤖 ML Connection — What `.fit()` Does Inside the Pipeline

When you call `pipe.fit(X_train, y_train)`, the pipeline:
1. Calls `preprocessor.fit_transform(X_train)` — fits the scalers and encoders on training data, returns transformed array
2. Calls `model.fit(transformed_X_train, y_train)` — fits the classifier

When you call `pipe.predict(X_test)`, it:
1. Calls `preprocessor.transform(X_test)` — applies training parameters, does NOT refit
2. Calls `model.predict(transformed_X_test)` — classifies

Step 1 of predict uses `.transform()`, not `.fit_transform()`. That is the leakage prevention — enforced automatically, every time.

---

### ✏️ Exercise 1.1 — Access Pipeline Internals

1. Access the fitted preprocessor via `pipe.named_steps['preprocessor']` and retrieve the output feature names.
2. Access the fitted model via `pipe.named_steps['model']` and build a feature importance DataFrame, sorted descending.
3. Run 5-fold CV using `cross_val_score` on the full pipeline — pass raw `X_train` and `y_train`. Print mean and std F1.
4. Compare this CV mean to Week 6's untuned random forest CV mean. Write a comment: should they match, and why?

In [ ]:
# Exercise 1.1

# 1. Feature names from the fitted preprocessor
fitted_pre   = pipe.named_steps['preprocessor']
feature_names = 
print('Feature names from pipeline preprocessor:')
print(feature_names)

# 2. Feature importances from the fitted model
fitted_model = pipe.named_steps['model']
importances  = pd.DataFrame({
    'feature':    feature_names,
    'importance': # your code here
}).sort_values('importance', ascending=False)

print('\nTop 5 feature importances:')
print(importances.head(5).to_string(index=False))

# 3. Cross-validation on the full pipeline
pipe_scores = cross_val_score(
    pipe, X_train, y_train,   # raw X_train — pipeline handles transform
    cv=5, scoring='f1'
)
print(f'\nPipeline CV F1 — mean: {pipe_scores.mean():.3f}  std: {pipe_scores.std():.3f}')

# 4. Compare to Week 6
# Week 6 untuned RF CV mean was approximately: ___
# Comment — should they match?
# 

---
## Part 2 — Pipeline + GridSearchCV

Pass the pipeline directly to `GridSearchCV`. Hyperparameter names use the `stepname__parametername` convention — two underscores.

In [ ]:
# Grid search on the full pipeline
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth':    [5, 10, None],
}

grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

# Fit on raw X_train — the pipeline inside GridSearchCV handles transformation
grid_search.fit(X_train, y_train)

print(f'Best parameters:  {grid_search.best_params_}')
print(f'Best CV F1:       {grid_search.best_score_:.3f}')

# Predict with the best pipeline
y_pred_tuned = grid_search.best_estimator_.predict(X_test)

print('\nTuned pipeline — Test Set:')
print(classification_report(y_test, y_pred_tuned, target_names=['Retained', 'Churned']))

---
### 💼 Business Context — Tuning the Whole Workflow, Not Just the Model

In the Week 6 grid search, `GridSearchCV` received already-processed data — the preprocessor was fixed and outside the search. Here, the full pipeline is inside `GridSearchCV`. This means the preprocessing and model are evaluated together as a unit in every fold, which is the only correct way to assess whether a combination of preprocessing choices and model hyperparameters generalizes. If you ever wanted to also tune preprocessing parameters (e.g., whether to use `StandardScaler` or `MinMaxScaler`), you could add them to `param_grid` using the full prefix: `'preprocessor__num__copy': [True]`.

---

### ✏️ Exercise 2.1 — Extend and Interpret the Grid

1. Add `'model__min_samples_leaf': [1, 3, 5]` to `param_grid` and re-run `GridSearchCV`. Print the new best parameters and best CV F1.
2. Print the top 5 results from `grid_search.cv_results_` — show `params`, `mean_test_score`, `std_test_score`.
3. Calculate the total number of model fits performed in the extended grid search.
4. Compare the tuned pipeline's test F1 to the untuned pipeline's test F1 from Part 1. Write a comment on whether the improvement justifies the added search time.

In [ ]:
# Exercise 2.1

# 1. Extended grid
param_grid_ext = {
    'model__n_estimators':     [50, 100, 200],
    'model__max_depth':        [5, 10, None],
    'model__min_samples_leaf': # your values here
}

grid_search_ext = GridSearchCV(
    pipe, param_grid_ext,
    cv=5, scoring='f1', n_jobs=-1
)
grid_search_ext.fit(X_train, y_train)

print(f'Extended grid — best parameters: {grid_search_ext.best_params_}')
print(f'Extended grid — best CV F1:      {grid_search_ext.best_score_:.3f}')

# 2. Top 5 results
results_df = pd.DataFrame(grid_search_ext.cv_results_)
print('\nTop 5 results:')
print(
    results_df[['params', 'mean_test_score', 'std_test_score']]
    .sort_values('mean_test_score', ascending=False)
    .head(5)
    .round(4)
    .to_string(index=False)
)

# 3. Total fits
total_fits = 
print(f'\nTotal fits: {total_fits}')

# 4. Tuned vs. untuned test F1
untuned_f1 = f1_score(y_test, y_pred)          # provided — do not change
tuned_f1   = f1_score(y_test, y_pred_tuned)    # provided — do not change
ext_pred   = grid_search_ext.best_estimator_.predict(X_test)
ext_f1     = f1_score(y_test, ext_pred)

print(f'Untuned pipeline test F1:       {untuned_f1:.3f}')
print(f'Tuned pipeline test F1:         {tuned_f1:.3f}')
print(f'Extended-grid tuned test F1:    {ext_f1:.3f}')
# Comment:
# 

---
## Part 3 — Risk Scores and Customer Segmentation

Predicted class labels (0 or 1) tell you who the model thinks will churn. Predicted probabilities tell you how confident the model is — and let you rank customers by risk. Risk tiers convert that ranking into actionable business segments.

In [ ]:
# Use the best pipeline from the original grid search
best_pipe = grid_search.best_estimator_

# Predicted churn probability for each test customer
churn_proba = best_pipe.predict_proba(X_test)[:, 1]

# Build a results table
results = X_test.copy().reset_index(drop=True)
results['churn_probability'] = churn_proba
results['predicted_churned'] = (churn_proba >= 0.5).astype(int)
results['actual_churned']    = y_test.values

# Segment into risk tiers
results['risk_tier'] = pd.cut(
    churn_proba,
    bins=[0, 0.25, 0.50, 0.75, 1.01],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print('Risk tier distribution:')
print(results['risk_tier'].value_counts().sort_index())

print('\nSample — top 10 highest-risk customers:')
print(
    results.nlargest(10, 'churn_probability')
    [['tenure_months', 'contract_type', 'monthly_charges', 'churn_probability', 'risk_tier']]
    .to_string(index=False)
)

---
### 🤖 ML Connection — Probability Calibration

A predicted probability of 0.73 means the model assigns 73% likelihood of churn. For this to be meaningful, the model must be **calibrated** — among all customers predicted at 0.73, approximately 73% should actually churn. Random forests are not perfectly calibrated by default (they tend to push probabilities toward extremes), but they are sufficiently calibrated for ranking purposes. If exact probability estimates matter (e.g., for pricing risk), apply `CalibratedClassifierCV` as a wrapper. For churn prioritization — who to call first — uncalibrated probabilities are fine.

---

### ✏️ Exercise 3.1 — Validate the Risk Tiers

1. Compute the actual churn rate within each risk tier. Print a table showing: risk tier, customer count, predicted churn rate (from labels), and actual churn rate.
2. Does the actual churn rate increase monotonically from Low to Critical? Write a comment — what would it mean if a higher risk tier had a lower actual churn rate?
3. Calculate the precision and recall for the Critical tier specifically: treat Critical as positive, all others as negative.

In [ ]:
# Exercise 3.1

# 1. Churn rate by risk tier
tier_summary = results.groupby('risk_tier', observed=True).agg(
    count            = ('actual_churned', 'count'),
    predicted_churn  = ('predicted_churned', 'mean'),
    actual_churn     = ('actual_churned', 'mean'),
).round(3)

print('Risk tier validation:')
print(tier_summary)

# 2. Monotonically increasing?
# 

# 3. Precision and recall for Critical tier
results['is_critical'] = (results['risk_tier'] == 'Critical').astype(int)

critical_precision = 
critical_recall    = 

print(f'\nCritical tier precision: {critical_precision:.3f}')
print(f'Critical tier recall:    {critical_recall:.3f}')
# Interpretation:
# 

---
## Part 4 — Communicating Results with Plotly Express

The model output has been shaped into a results table. Now present it. Each chart answers a different business question.

In [ ]:
# Chart 1 — Churn probability distribution by risk tier
fig1 = px.histogram(
    results,
    x='churn_probability',
    color='risk_tier',
    nbins=25,
    title='Predicted Churn Probability — Risk Tier Distribution',
    labels={'churn_probability': 'Predicted Churn Probability', 'risk_tier': 'Risk Tier'},
    category_orders={'risk_tier': ['Low', 'Medium', 'High', 'Critical']},
    color_discrete_map={
        'Low': '#2166ac', 'Medium': '#92c5de',
        'High': '#f4a582', 'Critical': '#d6604d'
    }
)
fig1.show()

# Chart 2 — Actual churn rate by contract type (from full dataset)
churn_by_contract = (
    df.groupby('contract_type')['churned']
    .mean()
    .reset_index()
    .rename(columns={'churned': 'churn_rate'})
    .sort_values('churn_rate', ascending=False)
)

fig2 = px.bar(
    churn_by_contract,
    x='contract_type', y='churn_rate',
    title='Actual Churn Rate by Contract Type',
    labels={'churn_rate': 'Churn Rate', 'contract_type': 'Contract Type'},
    color='churn_rate',
    color_continuous_scale='Reds',
    text_auto='.1%'
)
fig2.update_layout(yaxis_tickformat='.0%', showlegend=False)
fig2.show()

---
### 💼 Business Context — Choosing the Right Chart for the Audience

The histogram (Chart 1) is for a data-literate audience — it shows the model's probability distribution and how risk tiers are defined. The bar chart (Chart 2) is for any audience — it shows a simple comparison that drives directly to the business recommendation: 'month-to-month customers churn at X% vs. Y% for annual contracts.' Lead with the bar chart in a presentation. Use the histogram to support methodology questions.

---

### ✏️ Exercise 4.1 — Build Two Presentation Charts

1. Create a `px.scatter` chart with `tenure_months` on the x-axis, `churn_probability` on the y-axis, colored by `contract_type`. Add a horizontal dashed line at y = 0.5 (the default threshold). Include `monthly_charges` and `risk_tier` as hover data.
2. Create a `px.box` chart showing the distribution of `churn_probability` for each `customer_segment`. Color by segment.
3. For each chart, write one sentence describing what it shows and one sentence on the business action it suggests.

In [ ]:
# Exercise 4.1

# 1. Scatter — tenure vs. churn probability, colored by contract type
fig3 = px.scatter(
    results,
    x=         ,
    y=         ,
    color=     ,
    hover_data=,
    title='Tenure vs. Predicted Churn Probability',
    labels={
        'tenure_months':     'Tenure (months)',
        'churn_probability':  'Predicted Churn Probability',
        'contract_type':      'Contract Type',
    }
)
fig3.add_hline(y=0.5, line_dash='dash', line_color='black',
               annotation_text='Decision threshold (0.5)')
fig3.show()

# What does this chart show?
# 
# Business action:
# 

# 2. Box — churn probability by customer segment
fig4 = px.box(
    results,
    x=     ,
    y=     ,
    color= ,
    title='Predicted Churn Probability by Customer Segment',
    labels={
        'customer_segment':  'Customer Segment',
        'churn_probability': 'Predicted Churn Probability',
    }
)
fig4.show()

# What does this chart show?
# 
# Business action:
# 

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

### Model Card

A model card is a brief structured document that accompanies a deployed model. It records what the model does, how it performs, what it should and should not be used for, and who is accountable for it.

Using Markdown cells below, write a model card for the churn pipeline you built this week. Include:

1. **Model name and version** — a short identifier and today's date
2. **Intended use** — what business decision this model supports; what it should NOT be used for
3. **Training data** — rows, features, target, time period (synthetic here, but describe as if real)
4. **Performance** — CV F1, test F1, test precision, and test recall for the Churned class. State the decision threshold used.
5. **Key drivers** — top 3 features by importance and a plain-language description of each
6. **Known limitations** — at least two honest limitations of this model
7. **Recommended action by risk tier** — one sentence per tier describing what the business should do

---

## Model Card — Churn Prediction Pipeline

**Model name:** ___ &nbsp;&nbsp; **Version:** 1.0 &nbsp;&nbsp; **Date:** ___

---

### 1. Intended Use

_Fill in: what business question this answers. What it should NOT be used for._

---

### 2. Training Data

_Fill in: rows, features, target, time period._

---

### 3. Performance

| Metric | Value |
|:-------|:------|
| CV F1 (5-fold) | ___ |
| Test F1 | ___ |
| Test Precision (Churned) | ___ |
| Test Recall (Churned) | ___ |
| Decision threshold | 0.50 |

---

### 4. Key Drivers

1. **___** — ___
2. **___** — ___
3. **___** — ___

---

### 5. Known Limitations

- ___
- ___

---

### 6. Recommended Action by Risk Tier

| Risk Tier | Recommended Action |
|:----------|:-------------------|
| Critical | ___ |
| High | ___ |
| Medium | ___ |
| Low | ___ |

---
## Week 7 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| `Pipeline(steps=[...])` | Chains preprocessing + model — enforces correct fit/transform sequence automatically |
| `pipe.fit(X_train, y_train)` | Fits all steps in order — no separate `fit_transform` calls needed |
| `pipe.predict(X_test)` | Transforms using training parameters, then classifies — leakage-safe by design |
| `named_steps` | Access any fitted pipeline component by step name |
| `step__param` | Two underscores — required for GridSearchCV on a pipeline |
| `predict_proba` | Returns probabilities — more useful than labels for ranking and prioritization |
| `pd.cut` | Converts continuous probabilities into labeled tiers |
| Risk tier validation | Actual churn rate should increase monotonically — verify before presenting |
| `px.histogram` | Probability distributions — for data-literate audiences |
| `px.bar` | Categorical comparisons — clear for any audience |
| `px.scatter` | Two numeric variables + color — reveals interaction patterns |
| `px.box` | Distribution by group — compares spread and median across segments |
| Model card | Documents what the model does, how it performs, and what it should not be used for |

---
**Next week:** Unsupervised learning — KMeans and hierarchical clustering for market segmentation. The target variable disappears. Instead of predicting a label, you will discover natural groupings in data where no labels exist.